# Segmentation Model

In [1]:
# First, ensure the correct library version is installed
# If you haven't installed it yet, run: !pip install ultralytics==8.3.0

from ultralytics import YOLO

# Load a pre-trained segmentation model
model = YOLO('yolov8n-seg.pt')

# Train the model with the specified arguments
# All the command-line arguments are passed as keyword arguments (kwargs)
results = model.train(
    data="fish_seg.yaml",
    imgsz=640,
    epochs=100,
    batch=8,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10,
    translate=0.1,
    scale=0.5,
    shear=5,
    device='0'  # Use 'cpu' or '0' for GPU if available
)

New https://pypi.org/project/ultralytics/8.3.205 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.203  Python-3.11.3 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3050 Ti Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=fish_seg.yaml, degrees=10, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, 

# Indian Species Model

In [2]:
# D:\Projects\MATSYA\codes\train_species_cls_gpu.py
import os, random
from pathlib import Path
import torch, torch.nn as nn
from torch.utils.data import DataLoader, random_split, WeightedRandomSampler
from torchvision import datasets, transforms, models

# ---------------- cfg ----------------
DATA = r"DATA/species_crops"
OUT  = r"models/species_cls"
IMG = 224
BATCH = 8          # 8 if you see CUDA OOM; 32 if your RAM/GPU allow
VAL_FRAC = 0.15
EPOCHS_WARM = 3
EPOCHS_FT = 12
SEED = 42
NUM_WORKERS = 4     # increase if CPU is strong
# ------------------------------------

os.makedirs(OUT, exist_ok=True)

# ---- GPU setup & perf knobs ----
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    torch.backends.cudnn.benchmark = True   # autotune convs
    torch.set_float32_matmul_precision("high")  # Ampere+ speed
# --------------------------------

# ---- Repro ----
torch.manual_seed(SEED); random.seed(SEED)

# ---- Transforms ----
tfm_train = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2,0.2,0.2,0.05),
    transforms.RandomAffine(degrees=12, translate=(0.08,0.08), scale=(0.9,1.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
tfm_val = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# ---- Dataset & split ----
full = datasets.ImageFolder(DATA, transform=tfm_train)
num_classes = len(full.classes)
n_val = int(len(full)*VAL_FRAC)
n_train = len(full)-n_val
train_ds, val_ds = random_split(full, [n_train, n_val])

# ---- Class imbalance → weighted sampler ----
# Count per-class from an "index-only" dataset so we don't decode images here.
tmp = datasets.ImageFolder(DATA)  # no transform loads paths/targets
train_idx = set(train_ds.indices)
counts = [0]*num_classes
for i, (_, lab) in enumerate(tmp.imgs):
    if i in train_idx: counts[lab] += 1
class_weights = [0. if c == 0 else 1.0/c for c in counts]
sample_weights = []
for i, (_, lab) in enumerate(tmp.imgs):
    if i in train_idx: sample_weights.append(class_weights[lab])
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

# ---- DataLoaders ----
train_loader = DataLoader(
    train_ds, batch_size=BATCH, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=(device=="cuda"),
    persistent_workers=(NUM_WORKERS>0)
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(device=="cuda"),
    persistent_workers=(NUM_WORKERS>0)
)

# ---- Model ----
model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)

# Memory-friendly settings
if device == "cuda":
    model = model.to(device, memory_format=torch.channels_last)
else:
    model = model.to(device)

# ---- Optim & loss ----
crit = nn.CrossEntropyLoss()

# AMP scaler
scaler = torch.cuda.amp.GradScaler(enabled=(device=="cuda"))

def run_epoch(loader, train=True, opt=None):
    model.train(train)
    total, correct, loss_sum = 0, 0, 0.0
    for x, y in loader:
        if device == "cuda":
            x = x.to(device, non_blocking=True).to(memory_format=torch.channels_last)
        else:
            x = x.to(device)
        y = y.to(device, non_blocking=(device=="cuda"))

        with torch.cuda.amp.autocast(enabled=(device=="cuda")):
            out = model(x)
            loss = crit(out, y)

        if train:
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

        loss_sum += loss.item() * y.size(0)
        pred = out.argmax(1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return loss_sum/total, correct/total

# ---- Warmup: freeze backbone ----
for p in model.features.parameters(): p.requires_grad = False
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)

for ep in range(1, EPOCHS_WARM+1):
    tr = run_epoch(train_loader, True, opt)
    va = run_epoch(val_loader, False)
    print(f"[warm {ep}] train_loss={tr[0]:.4f} acc={tr[1]:.3f} | val_loss={va[0]:.4f} acc={va[1]:.3f}")

# ---- Finetune all ----
for p in model.features.parameters(): p.requires_grad = True
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)

best_acc, best_path = 0.0, os.path.join(OUT, "species_mnv3_small.pt")

for ep in range(1, EPOCHS_FT+1):
    tr = run_epoch(train_loader, True, opt)
    va = run_epoch(val_loader, False)
    print(f"[ft {ep}] train_loss={tr[0]:.4f} acc={tr[1]:.3f} | val_loss={va[0]:.4f} acc={va[1]:.3f}")
    if va[1] > best_acc:
        best_acc = va[1]
        torch.save({"model": model.state_dict(), "classes": full.classes}, best_path)
        print("saved:", best_path, "acc:", best_acc)

# also save a final checkpoint
torch.save({"model": model.state_dict(), "classes": full.classes},
           os.path.join(OUT, "species_mnv3_small_last.pt"))
print("done. best val acc:", best_acc)


Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to C:\Users\rupes/.cache\torch\hub\checkpoints\mobilenet_v3_small-047dcff4.pth
100%|██████████| 9.83M/9.83M [00:02<00:00, 4.30MB/s]
C:\Users\rupes\AppData\Local\Temp\ipykernel_22504\1338840987.py:93: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device=="cuda"))
C:\Users\rupes\AppData\Local\Temp\ipykernel_22504\1338840987.py:105: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device=="cuda")):


[warm 1] train_loss=0.3625 acc=0.891 | val_loss=0.1966 acc=0.939
[warm 2] train_loss=0.1912 acc=0.942 | val_loss=0.1380 acc=0.957
[warm 3] train_loss=0.1859 acc=0.941 | val_loss=0.1495 acc=0.956
[ft 1] train_loss=0.1046 acc=0.966 | val_loss=0.0914 acc=0.970
saved: models/species_cls\species_mnv3_small.pt acc: 0.9704797047970479
[ft 2] train_loss=0.0702 acc=0.978 | val_loss=0.1003 acc=0.970
[ft 3] train_loss=0.0453 acc=0.987 | val_loss=0.0857 acc=0.974
saved: models/species_cls\species_mnv3_small.pt acc: 0.974169741697417
[ft 4] train_loss=0.0616 acc=0.981 | val_loss=0.1086 acc=0.973
[ft 5] train_loss=0.0291 acc=0.991 | val_loss=0.0702 acc=0.984
saved: models/species_cls\species_mnv3_small.pt acc: 0.9837638376383764
[ft 6] train_loss=0.0480 acc=0.986 | val_loss=0.0816 acc=0.978
[ft 7] train_loss=0.0402 acc=0.988 | val_loss=0.0613 acc=0.981
[ft 8] train_loss=0.0258 acc=0.992 | val_loss=0.0666 acc=0.978
[ft 9] train_loss=0.0377 acc=0.989 | val_loss=0.0639 acc=0.978
[ft 10] train_loss=0.02

# Fish Freshness Model

In [ ]:

9

fixed/flagged: 17


In [4]:
from ultralytics import YOLO

# Initialize the model (YOLOv8n for detection, pre-trained weights)
model = YOLO('yolov8n.pt')

# Define the dataset configuration path (using r-prefix for clean Windows paths)
data_path = r"data/Fish Freshness-eyes-skin/data.yaml"

# Train the model
results = model.train(
    data=data_path,
    imgsz=640,
    epochs=100,
    batch=16,
    name='fish_freshness_yolov8',
    workers=0,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    shear=5.0,
    device='0',  # Use 'cpu' or '0' for GPU if available
    plots=True, # Optional: Add this to save confusion matrix and other plots
)

New https://pypi.org/project/ultralytics/8.3.205 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.203  Python-3.11.3 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3050 Ti Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/Fish Freshness-eyes-skin/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=f

In [7]:
!yolo export model=runs/detect/fish_freshness_yolov85/weights/best.pt format=onnx

Ultralytics 8.3.203  Python-3.11.3 torch-2.6.0+cu124 CPU (AMD Ryzen 7 4800H with Radeon Graphics)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'runs\detect\fish_freshness_yolov85\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 10, 8400) (6.0 MB)

ONNX: starting export with onnx 1.19.0 opset 19...
ONNX: slimming with onnxslim 0.1.70...
ONNX: export success  3.2s, saved as 'runs\detect\fish_freshness_yolov85\weights\best.onnx' (11.7 MB)

Export complete (3.7s)
Results saved to D:\Projects\MATSYA\Codes\runs\detect\fish_freshness_yolov85\weights
Predict:         yolo predict task=detect model=runs\detect\fish_freshness_yolov85\weights\best.onnx imgsz=640  
Validate:        yolo val task=detect model=runs\detect\fish_freshness_yolov85\weights\best.onnx imgsz=640 data=data/Fish Freshness-eyes-skin/data.yaml  
Visualize:       https://netron.app
 Learn more at https://docs.ultralytics.com/modes/export


In [27]:
import pandas as pd, numpy as np
from pathlib import Path

# INPUT: your scraped FishBase table (TSV/CSV) with columns like in your sample
IN  = r"excel_fishbase_lw.xlsx"
OUT = r"assets/fishbase_lwr.csv"

# --------- CONFIG (edit if needed) ---------
# Acceptable range for b and "K at 30 cm" (crude sanity filters)
B_MIN, B_MAX = 2.5, 3.5
K30_MIN, K30_MAX = 0.4, 2.5    # typical Fulton K @ 30cm (unitless)
PREFER_SEX_ORDER = ["unsexed", "mixed", "female", "male"]  # your call
# -------------------------------------------

def as_float(x):
    try:
        return float(str(x).strip())
    except:
        return np.nan

def normalize_columns(df):
    # rename flexible columns to canonical names we’ll use
    ren = {
        "SpeciesID":"SpeciesID",
        "Species":"Species",
        "a":"a",
        "b":"b",
        "Sex":"Sex",
        "n":"n",
        "Doubtful?":"Doubtful",
        "Length (cm)":"LengthRange",
        "Territory":"Territory",
        "Locality":"Locality",
        "Method":"Method",
        "SpeciesLWURL":"SpeciesLWURL",
        "SourceDetailURL":"SourceDetailURL",
    }
    # keep only those we recognize
    keep = {k:v for k,v in ren.items() if k in df.columns}
    df = df.rename(columns=keep)[list(keep.values())]
    return df

def compute_k30(a,b):
    # Fulton K ~ 100*W/L^3 ; W = a L^b with L in cm, W in g
    # Evaluate at L=30 cm to avoid scale sensitivity: K30 = 100 * a * 30^(b-3)
    return 100.0 * a * (30.0 ** (b - 3.0))

def score_row(row):
    # Higher is better: not doubtful, larger n, b near 3, sane K30, preferred sex
    s = 0.0
    # not doubtful preferred
    if str(row.get("Doubtful","")).strip().lower() in ("no","","nan"):
        s += 3.0
    # more samples better
    n = row.get("n", np.nan)
    if pd.notna(n): s += min(2.0, np.log10(max(1.0, float(n))) )
    # b near 3 better
    b = row["b"]
    if pd.notna(b):
        s += max(0.0, 1.5 - abs(float(b) - 3.0))   # within ±1.5 gets positive
    # K30 within band
    K30 = row.get("K30", np.nan)
    if pd.notna(K30):
        if K30_MIN <= K30 <= K30_MAX: s += 1.5
    # sex preference
    sex = str(row.get("Sex","")).strip().lower()
    if sex in PREFER_SEX_ORDER:
        s += max(0, 1.0 - 0.25*PREFER_SEX_ORDER.index(sex))
    return s

# ---- load & basic clean
raw = pd.read_excel(IN)
raw = normalize_columns(raw)

# coerce numeric
raw["a"] = raw["a"].apply(as_float)
raw["b"] = raw["b"].apply(as_float)
if "n" in raw.columns:
    raw["n"] = raw["n"].apply(as_float)
else:
    raw["n"] = np.nan

# basic validity
df = raw.dropna(subset=["Species","a","b"]).copy()

# compute K30 and keep as diagnostic
df["K30"] = compute_k30(df["a"].astype(float), df["b"].astype(float))

# filter obviously bad rows (unit mistakes, etc.)
good = df.copy()
good = good[(good["b"] >= B_MIN) & (good["b"] <= B_MAX)]
good = good[(good["K30"] >= K30_MIN) & (good["K30"] <= K30_MAX)]

# If a species has no "good" rows, fall back to looser filter just on b
fallback_species = set(df["Species"]) - set(good["Species"])
fallback = df[df["Species"].isin(fallback_species)]
fallback = fallback[(fallback["b"] >= 2.3) & (fallback["b"] <= 3.7)]

combined = pd.concat([good, fallback], ignore_index=True).drop_duplicates()

# score & pick best per species
combined["__score"] = combined.apply(score_row, axis=1)
best = (combined.sort_values(["Species","__score"], ascending=[True,False])
               .groupby("Species", as_index=False).first())

# final CSV for the pipeline
out = best[["Species","a","b","n","Sex","K30","Territory","Locality","Method","SpeciesLWURL","SourceDetailURL"]].copy()
out.insert(1, "common_name", best["Species"])
out.insert(2, "length_type", "TL")   # FishBase is often TL; if you know FL/SL, change here per species
out = out.rename(columns={
    "Species":"species",
    "common_name":"common_name",
    "length_type":"length_type",
    "a":"a","b":"b","n":"n"
})

Path(OUT).parent.mkdir(parents=True, exist_ok=True)
out.to_csv(OUT, index=False)
print(f"Saved {OUT} with {len(out)} species")
# Also dump a quick report of dropped species
dropped = sorted(set(raw["Species"]) - set(out["species"]))
print("Dropped (no usable record):", len(dropped))


Saved assets/fishbase_lwr.csv with 7285 species
Dropped (no usable record): 85


In [8]:
# make_aruco_50mm.py
import cv2, numpy as np

MM = 50.0      # physical size
DPI = 300      # print DPI
px = int(round(MM/25.4 * DPI))

aru = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
img = cv2.aruco.generateImageMarker(aru, 23, px)  # ID=23
pad = int(0.15*px)
canvas = 255*np.ones((px+2*pad, px+2*pad), np.uint8)
canvas[pad:pad+px, pad:pad+px] = img
cv2.imwrite("aruco_50mm_id23_300dpi.png", canvas)
print("Saved aruco_50mm_id23_300dpi.png (print at 100% scale).")


Saved aruco_50mm_id23_300dpi.png (print at 100% scale).


1) Calibration (multi-tier px/mm or direct 3D)

In [29]:
# calibrate.py
import cv2, numpy as np
from typing import Optional, Tuple

# ---------- Tier 1: Depth 3D ----------
def length_3d_mm(p1, p2, depth_mm, fx, fy, cx, cy):
    """
    p1, p2: (u,v) pixel endpoints along the fish major axis
    depth_mm: HxW uint16 (mm) or float depth map (mm)
    fx,fy,cx,cy: camera intrinsics
    """
    def backproject(u,v,z):
        X = (u - cx) * z / fx
        Y = (v - cy) * z / fy
        return np.array([X, Y, z], dtype=np.float32)

    z1 = float(depth_mm[int(round(p1[1])), int(round(p1[0]))])
    z2 = float(depth_mm[int(round(p2[1])), int(round(p2[0]))])
    if z1 <= 0 or z2 <= 0:
        return None, 0.0  # invalid depth
    P1 = backproject(p1[0], p1[1], z1)
    P2 = backproject(p2[0], p2[1], z2)
    Lmm = float(np.linalg.norm(P1 - P2))
    # quality: lower depth variance => better
    q = 0.9
    return Lmm, q

# ---------- Tier 2: ArUco ----------
def aruco_px_per_mm(bgr, marker_mm=50.0, dict_name=cv2.aruco.DICT_4X4_50, wanted_id=None):
    ar = cv2.aruco
    d = ar.getPredefinedDictionary(dict_name)
    det = ar.ArucoDetector(d, ar.DetectorParameters())
    corners, ids, _ = det.detectMarkers(bgr)
    if ids is None: return None, 0.0
    ids = ids.flatten()
    side_list = []
    for i,cid in enumerate(ids):
        if wanted_id is not None and int(cid) != wanted_id: continue
        pts = corners[i].reshape(-1,2)
        s = 0.0
        for j in range(4):
            a,b = pts[j], pts[(j+1)%4]
            s += np.linalg.norm(a-b)
        side_px = s/4.0
        side_list.append(side_px)
    if not side_list: return None, 0.0
    side_px = float(np.median(side_list))
    px_per_mm = side_px / float(marker_mm)
    # quality grows with marker size in px
    q = min(1.0, side_px / 150.0)  # ~150px side => high
    return px_per_mm, q

# ---------- Tier 3: Known planar rectangle (A4 / card) ----------
def quad_px_per_mm_from_known_rect(bgr, known_w_mm, known_h_mm, aspect_tol=0.08):
    g = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    g = cv2.GaussianBlur(g, (5,5), 0)
    edges = cv2.Canny(g, 60, 150)
    cnts,_ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    best = None; best_q = 0.0
    for c in cnts:
        peri = cv2.arcLength(c, True)
        approx = cv2.approxPolyDP(c, 0.02*peri, True)
        if len(approx) != 4: continue
        rect = cv2.minAreaRect(approx)
        (w,h) = rect[1]
        if w < 40 or h < 40: continue
        asp = max(w,h)/max(1,h if w>h else w)
        exp = max(known_w_mm, known_h_mm) / max(1, min(known_w_mm, known_h_mm))
        if abs(asp - exp) > exp*aspect_tol: continue
        px_per_mm_w = w / known_w_mm
        px_per_mm_h = h / known_h_mm
        px_per_mm = 0.5*(px_per_mm_w + px_per_mm_h)
        q = min(1.0, min(w,h)/200.0)
        if q > best_q:
            best_q = q; best = px_per_mm
    if best is None: return None, 0.0
    return float(best), float(best_q)

# ---------- Tier 4: Manual scale ----------
def manual_px_per_mm(p1, p2, known_len_mm):
    dpx = float(np.linalg.norm(np.array(p1, dtype=np.float32) - np.array(p2, dtype=np.float32)))
    if dpx <= 0: return None, 0.0
    return dpx/float(known_len_mm), 0.6

# ---------- Orchestrator ----------
def calibrate_px_or_length(
    bgr,
    # depth inputs (if any)
    depth_mm: Optional[np.ndarray]=None, intrinsics: Optional[Tuple[float,float,float,float]]=None, depth_endpoints: Optional[Tuple[tuple,tuple]]=None,
    # aruco
    use_aruco=True, aruco_mm=50.0, aruco_dict=cv2.aruco.DICT_4X4_50, aruco_id=None,
    # known rectangles
    use_a4=False, use_card=False,
    # manual
    manual_points: Optional[Tuple[tuple,tuple]]=None, manual_len_mm: Optional[float]=None
):
    # Tier 1: depth 3D
    if depth_mm is not None and intrinsics is not None and depth_endpoints is not None:
        Lmm, q = length_3d_mm(depth_endpoints[0], depth_endpoints[1], depth_mm, *intrinsics)
        if Lmm is not None:
            return {"mode":"depth", "length_mm":Lmm, "px_per_mm":None, "q":q}

    # Tier 2: ArUco
    if use_aruco:
        pxmm, q = aruco_px_per_mm(bgr, aruco_mm, aruco_dict, aruco_id)
        if pxmm:
            return {"mode":"aruco", "length_mm":None, "px_per_mm":pxmm, "q":q}

    # Tier 3: A4 / card
    if use_a4:
        pxmm, q = quad_px_per_mm_from_known_rect(bgr, 210.0, 297.0)  # either orientation ok
        if pxmm:
            return {"mode":"a4", "length_mm":None, "px_per_mm":pxmm, "q":q}
    if use_card:
        pxmm, q = quad_px_per_mm_from_known_rect(bgr, 85.6, 53.98)
        if pxmm:
            return {"mode":"card", "length_mm":None, "px_per_mm":pxmm, "q":q}

    # Tier 4: manual
    if manual_points and manual_len_mm:
        pxmm, q = manual_px_per_mm(*manual_points, manual_len_mm)
        if pxmm:
            return {"mode":"manual", "length_mm":None, "px_per_mm":pxmm, "q":q}

    # Tier 5: species prior (handled later, needs species + length_px)
    return {"mode":"none", "length_mm":None, "px_per_mm":None, "q":0.0}


2) Length/girth from mask (robust)

In [30]:
# shape_metrics.py
import numpy as np, cv2

def major_axis_and_rect(poly_xy):
    cnt = np.array(poly_xy, dtype=np.float32).reshape(-1,1,2)
    rect = cv2.minAreaRect(cnt)          # ((cx,cy),(w,h),angle)
    w_rect, h_rect = rect[1]
    Lpx = max(w_rect, h_rect)
    return Lpx, rect

def girth_mm_from_rect(rect, px_per_mm):
    w_rect, h_rect = rect[1]
    minor_px = min(w_rect, h_rect)
    return float(minor_px/px_per_mm) if px_per_mm else None

def endpoints_along_major(rect):
    (cx,cy),(w,h),ang = rect
    L = max(w,h); S = min(w,h)
    # unit vectors
    theta = np.deg2rad(ang)
    ux, uy = (np.cos(theta), np.sin(theta))
    if h > w:  # swap axis if needed
        ux, uy = -uy, ux
    p1 = (cx - 0.5*L*ux, cy - 0.5*L*uy)
    p2 = (cx + 0.5*L*ux, cy + 0.5*L*uy)
    return (p1[0],p1[1]), (p2[0],p2[1])


3) LWR + volume

In [31]:
# lwr_utils.py
import csv, math

def load_lwr(csv_path):
    tbl={}
    with open(csv_path, newline='', encoding='utf-8') as f:
        for r in csv.DictReader(f):
            tbl[r['common_name'].lower()] = {"a":float(r['a']), "b":float(r['b']), "len":r['length_type']}
    return tbl

def weight_volume_from_length(length_mm, common_name, lwr_tbl, density=1.04):
    key = common_name.lower()
    if key not in lwr_tbl: return None, None
    a = lwr_tbl[key]["a"]; b = lwr_tbl[key]["b"]
    L_cm = length_mm/10.0
    W_g = a*(L_cm**b)
    V_cm3 = W_g/density
    return float(W_g), float(V_cm3)

def propagate_weight_uncertainty(length_mm, sigma_L_mm, a, b):
    # σW/W ≈ b * σL/L
    if length_mm<=0: return None
    rel = abs(b) * (sigma_L_mm / length_mm)
    return rel


4) Orchestrator (tries tiers, handles species-prior fallback)

In [32]:
# measure_fish.py
import numpy as np
# from calibrate import calibrate_px_or_length
# from shape_metrics import major_axis_and_rect, girth_mm_from_rect, endpoints_along_major
# from lwr_utils import load_lwr, weight_volume_from_length, propagate_weight_uncertainty

def species_prior_pxmm(major_axis_px, expected_length_mm):
    if expected_length_mm and expected_length_mm>0:
        return major_axis_px/expected_length_mm, 0.2  # low confidence
    return None, 0.0

def measure_one_fish(
    bgr, poly_xy, species_common_name,
    lwr_tbl,                        # dict from load_lwr()
    depth_mm=None, intrinsics=None, # if you have ARCore depth path
    want_a4=False, want_card=False,
    manual_points=None, manual_len_mm=None,
    species_length_prior_mm=None     # optional prior median length per species
):
    Lpx, rect = major_axis_and_rect(poly_xy)
    p1, p2 = endpoints_along_major(rect)

    # Try calibration
    calib = calibrate_px_or_length(
        bgr, depth_mm, intrinsics, depth_endpoints=(p1,p2),
        use_aruco=True, aruco_mm=50.0,
        use_a4=want_a4, use_card=want_card,
        manual_points=manual_points, manual_len_mm=manual_len_mm
    )

    # Case A: Depth gave direct metric length
    if calib["mode"]=="depth" and calib["length_mm"] is not None:
        Lmm = calib["length_mm"]
        pxmm = None; q = calib["q"]
    else:
        pxmm = calib["px_per_mm"]
        q = calib["q"]
        if pxmm is None:
            # Tier 5: species prior
            pxmm, q = species_prior_pxmm(Lpx, species_length_prior_mm)
            if pxmm is None:
                return {"ok":False, "reason":"no scale available"}
        Lmm = float(Lpx/pxmm)

    # Girth (optional for geometric volume)
    Gmm = girth_mm_from_rect(rect, pxmm) if pxmm else None

    # Weight/Volume via LWR
    Wg, Vcm3 = weight_volume_from_length(Lmm, species_common_name, lwr_tbl)

    # Simple uncertainty estimate
    # sigma_pxmm from method quality -> rough mapping:
    sigma_scale = (1.0-q) * 0.1 * (pxmm if pxmm else 1.0)  # ~10% at q=0; 0% at q=1
    sigma_L = (Lpx * sigma_scale / (pxmm if pxmm else 1.0)) if pxmm else 3.0  # a few mm if depth
    a = lwr_tbl.get(species_common_name.lower(), {}).get("a", None)
    b = lwr_tbl.get(species_common_name.lower(), {}).get("b", None)
    rel_w = propagate_weight_uncertainty(Lmm, sigma_L, a if a else 1.0, b if b else 3.0) if (a and b) else None

    return {
        "ok": True,
        "scale_mode": calib["mode"],
        "q_scale": q,
        "length_mm": Lmm,
        "girth_mm": Gmm,
        "weight_g": Wg,
        "volume_cm3": Vcm3,
        "rel_weight_uncertainty": rel_w
    }


In [33]:
# sanity_aruco.py
import cv2
# from aruco_scale import detect_px_per_mm

img_path = r"aruco_50mm_id23_300dpi.png"
im = cv2.imread(img_path)
pxmm, q = aruco_px_per_mm(im, marker_mm=50.0)
print("px_per_mm:", pxmm, "quality:", q)


px_per_mm: 11.8 quality: 1.0


In [41]:
# example_run.py  (fixed)
import cv2, json
from ultralytics import YOLO
# from lwr_utils import load_lwr
# from measure_fish import measure_one_fish

SEG_MODEL = r"runs/segment/train/weights/best.pt"
LWR = r"assets/fishbase_lwr.csv"

img_path = r"data/species_crops/Rohu/Rohu-Body (2)_0.jpg"
species_name = "Labeo Rohita"  # from your species classifier result

im = cv2.imread(img_path)
r = YOLO(SEG_MODEL).predict(source=im, conf=0.25, imgsz=640, verbose=False)[0]
lwr_tbl = load_lwr(LWR)

items=[]
if r.masks is None:
    print("No fish detected. Ensure image has a fish and your seg model path is correct.")
else:
    for poly in r.masks.xy:
        out = measure_one_fish(
            im, poly, species_name, lwr_tbl,
            depth_mm=None, intrinsics=None,   # ARCore later
            want_a4=True, want_card=False,    # allow A4 fallback
            manual_points=None, manual_len_mm=None,
            species_length_prior_mm=350.0
        )
        items.append(out)

print(json.dumps(items, indent=2))


[
  {
    "ok": true,
    "scale_mode": "aruco",
    "q_scale": 0.26518468221028646,
    "length_mm": 6214.698960036963,
    "girth_mm": 1906.5284959385224,
    "weight_g": 4581323.548527374,
    "volume_cm3": 4405118.796660936,
    "rel_weight_uncertainty": 0.18408591084213413
  }
]
